# E3 — Collision-Probability (Pc) Recomputation Feasibility Spike

**Experiment ID:** `E3`. **Specification:** `EXPERIMENT_PLAN.md` §E3.
**Question:** does **Assumption A4** (`PROJECT_KNOWLEDGE.md` §9) hold — can Pc be recomputed from
the public CDM state/covariance fields closely enough to the reported `risk` to support the
label-noise sensitivity analysis (Contribution 2)?

This is the project's **#1 technical unknown** (risk R2) and the highest-stakes code in Phase 0.

**Pre-registration.** The agreement tolerance was written to `DECISIONS.md` — marked
*PROPOSED — awaiting Sidh's confirmation* — and encoded in `config/default.yaml`
(`pc_spike.tolerance`) **before** this comparison was run, per `CLAUDE.md` §3 and §10. This
notebook reads the tolerance from config; it does not choose one.

**Scope limit.** This notebook reports the measurement and characterises the failure modes. It
does **not** make the hold / partially-hold / fail call on A4 — that is Sidh's decision at the
Phase 0 checkpoint (`CLAUDE.md` §3, §11).

In [ ]:
# --- Setup, configuration, and provenance stamp ---------------------------------------------
import json, subprocess, sys
from datetime import datetime, timezone

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from kelvins_conformal.config import REPO_ROOT, load_config
from kelvins_conformal import data as kcdata
from kelvins_conformal.labelnoise import pc_foster as pf

cfg = load_config()
FIGDIR = cfg.path("figures_dir"); FIGDIR.mkdir(parents=True, exist_ok=True)
TABDIR = cfg.path("tables_dir"); TABDIR.mkdir(parents=True, exist_ok=True)

def git_sha() -> str:
    try:
        out = subprocess.run(["git", "rev-parse", "HEAD"], cwd=str(REPO_ROOT),
                             capture_output=True, text=True, check=True)
        return out.stdout.strip()
    except Exception:
        return "UNAVAILABLE (working tree is not a git repository)"

TOL = cfg.pc_spike.tolerance.abs_log10_risk
MINPROP = cfg.pc_spike.tolerance.min_proportion_within

PROVENANCE = {
    "experiment_ids": ["E3"],
    "git_commit_sha": git_sha(),
    "config_hash": cfg.config_hash,
    "seed": cfg.seed,
    "preregistered_tolerance_abs_log10": TOL,
    "preregistered_min_proportion_within": MINPROP,
    "tolerance_status": "PROPOSED — awaiting Sidh's confirmation (see DECISIONS.md)",
    "executed_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version.split()[0],
}
print(json.dumps(PROVENANCE, indent=2))

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIGDIR / f"{name}.{ext}", dpi=160, bbox_inches="tight")
    print(f"saved: reports/figures/{name}.png|pdf")

def save_table(df, name):
    df.to_csv(TABDIR / f"{name}.csv", index=True)
    print(f"saved: reports/tables/{name}.csv")

## 1. Statistical-core gate: the toy-geometry tests must pass *before* real data

`CLAUDE.md` §4: the Pc engine is unit-tested against analytically known cases before it touches
real data. This cell re-runs `tests/test_pc_foster.py` inside the notebook and **aborts the
notebook** if anything fails — so a rendered report can never contain a comparison produced by an
unvalidated engine.

In [ ]:
# --- Re-run the toy-geometry test suite as a hard gate --------------------------------------
res = subprocess.run([sys.executable, "-m", "pytest", "tests/test_pc_foster.py", "-q",
                      "--no-header", "-p", "no:cacheprovider"],
                     cwd=str(REPO_ROOT), capture_output=True, text=True)
print(res.stdout[-2500:])
if res.returncode != 0:
    print(res.stderr[-2000:])
    raise RuntimeError("Pc toy-geometry tests FAILED — refusing to run the real-data spike "
                       "(CLAUDE.md §4: statistical core is validated before use).")
print("GATE PASSED: the Pc engine reproduces its analytic toy geometries.")

In [ ]:
# --- Show the two analytic reference cases explicitly (they are the credibility anchor) -----
# Case 1: isotropic covariance, zero miss -> exact Rayleigh CDF, Pc = 1 - exp(-R^2 / 2 sigma^2).
# Case 2: isotropic covariance, offset miss, HBR << sigma -> first-order point-mass formula.
rows = []
for sigma, hbr in [(100.0, 5.0), (250.0, 50.0), (1000.0, 200.0)]:
    cov2d = np.diag([sigma**2, sigma**2])
    num = pf.pc_on_disk(np.array([0.0, 0.0]), cov2d, hbr, n_radial=400, n_angular=720)
    exact = pf.analytic_pc_isotropic_zero_miss(sigma, hbr)
    rows.append({"case": "isotropic / zero miss (exact Rayleigh)", "sigma [m]": sigma,
                 "HBR [m]": hbr, "miss [m]": 0.0, "numeric Pc": num, "analytic Pc": exact,
                 "rel. error": abs(num - exact) / exact})
for miss in [0.0, 50.0, 150.0, 300.0]:
    sigma, hbr = 500.0, 2.0
    cov2d = np.diag([sigma**2, sigma**2])
    num = pf.pc_on_disk(np.array([miss, 0.0]), cov2d, hbr, n_radial=300, n_angular=720)
    approx = pf.analytic_pc_isotropic_point_mass(sigma, hbr, miss)
    rows.append({"case": "isotropic / offset miss (point-mass limit)", "sigma [m]": sigma,
                 "HBR [m]": hbr, "miss [m]": miss, "numeric Pc": num, "analytic Pc": approx,
                 "rel. error": abs(num - approx) / approx})
toy = pd.DataFrame(rows).set_index("case")
display(toy); save_table(toy, "e3_toy_geometry_validation")
print(f"worst relative error against the analytic references: {toy['rel. error'].max():.2e}")

## 2. Pre-registered tolerance (read from config, restated verbatim)

From `DECISIONS.md`, entry *"E3 Pc-agreement tolerance (Assumption A4 / Q-LBL-01) — PROPOSED"*:

- **Per-CDM criterion:** `|log10(Pc_recomputed) − risk_reported| ≤ 0.5` log10 units.
- **Sample-level criterion:** at least **80%** of the stratified sample must satisfy it for A4 to
  be considered to *hold*. A 50–80% band, or agreement concentrated in a characterisable
  subpopulation, is reported as a *partial hold* — descriptively only.

The values printed below come from `config/default.yaml`, so the report and the code cannot drift
apart.

In [ ]:
print(f"pre-registered per-CDM tolerance : |Δ log10 risk| <= {TOL}")
print(f"pre-registered sample threshold   : >= {MINPROP:.0%} of the sample within tolerance")
print(f"sample size (config)              : {cfg.pc_spike.n_sample_cdms} CDMs")
print(f"risk strata (config)              : {cfg.pc_spike.n_risk_strata}, "
      f"edges {list(cfg.pc_spike.risk_strata_edges)}")
print(f"seed                              : {cfg.seed}")
print("\nSTATUS: PROPOSED — not yet confirmed by Sidh. Any hold/fail language below is\n"
      "conditional on this tolerance being accepted as written.")

## 3. Stratified sample and the fields the computation requires

The sample spans risk levels by design: equal numbers of CDMs are drawn from each pre-declared
stratum (the −30 floor atom, then the bands between the configured edges, ending in the
challenge's high-risk band). The strata come from config, fixed before this run — a quantile-based
scheme would have been swamped by the floor atom, which is ~40% of all CDM rows.

Sampling is from the **training split only** (the official test set is untouched, `CLAUDE.md` §2).

In [ ]:
# --- Missing-field prevalence for the fields Pc recomputation needs -------------------------
events = kcdata.load_events(cfg)
train = events[events["split"] == "train"].copy()
print(f"train CDM rows available: {len(train):,}")

missing_tbl = pd.DataFrame({
    "missing rate (train CDMs)": train[list(pf.REQUIRED_FIELDS)].isna().mean(),
    "missing count": train[list(pf.REQUIRED_FIELDS)].isna().sum().astype(int),
}).sort_values("missing rate (train CDMs)", ascending=False)
display(missing_tbl); save_table(missing_tbl, "e3_missing_field_prevalence")

n_required_missing = train[list(pf.REQUIRED_FIELDS)].isna().any(axis=1)
print(f"\ntrain CDMs missing >=1 required field: {int(n_required_missing.sum()):,} "
      f"({n_required_missing.mean():.4%})")
print("This answers the field-availability half of Q-DATA-05: the fields the Foster computation\n"
      "needs are essentially complete, so an 'M7 subset' would be nearly the whole dataset.")

In [ ]:
# --- Draw the stratified sample -------------------------------------------------------------
edges = list(cfg.pc_spike.risk_strata_edges)
floor = cfg.target.floor_sentinel_value

def assign_stratum(risk):
    """Stratum 0 = the floor atom; strata 1..k = the (edges[i-1], edges[i]] bands."""
    if np.isclose(risk, floor):
        return 0
    for i in range(1, len(edges)):
        if edges[i - 1] < risk <= edges[i]:
            return i
    return np.nan

train["stratum"] = train["risk"].map(assign_stratum)
labels = {0: f"floor (== {floor:g})"}
labels.update({i: f"({edges[i-1]:g}, {edges[i]:g}]" for i in range(1, len(edges))})

pop = train.groupby("stratum").size().rename("train CDMs in stratum").to_frame()
pop.index = [labels[int(i)] for i in pop.index]
display(pop)

rng = np.random.default_rng(cfg.seed)
per_stratum = int(np.ceil(cfg.pc_spike.n_sample_cdms / cfg.pc_spike.n_risk_strata))
parts = []
for s in sorted(train["stratum"].dropna().unique()):
    sub = train[train["stratum"] == s]
    take = min(per_stratum, len(sub))
    idx = rng.choice(sub.index.to_numpy(), size=take, replace=False)
    parts.append(train.loc[idx])
    print(f"stratum {labels[int(s)]:<20} population {len(sub):>7,}  sampled {take}")
sample = pd.concat(parts).sort_values("risk").reset_index(drop=True)
print(f"\ntotal sampled CDMs: {len(sample)} (target ~{cfg.pc_spike.n_sample_cdms})")

## 4. Recompute Pc and compare against the reported `risk`

In [ ]:
# --- Recompute per sampled CDM --------------------------------------------------------------
recs = []
for _, row in sample.iterrows():
    d = row.to_dict()
    entry = {"event_uid": d["event_uid"], "time_to_tca": d["time_to_tca"],
             "stratum": labels[int(d["stratum"])], "reported": float(d["risk"]),
             "reported_miss_distance": float(d.get("miss_distance", np.nan))}
    n_miss = pf.count_missing_required(d)
    if n_miss > 0:
        entry.update(recomputed=np.nan, status=f"missing {n_miss} required field(s)")
    else:
        try:
            r = pf.recompute_pc_for_row(d, floor_sentinel=floor)
            entry.update(recomputed=r.log10_pc, status="ok",
                         recomputed_miss_distance=r.miss_distance_2d, hbr=r.hbr)
        except pf.PcComputationError as e:
            entry.update(recomputed=np.nan, status=f"degenerate geometry: {e}")
    recs.append(entry)

comp = pd.DataFrame(recs)
comp["delta"] = comp["recomputed"] - comp["reported"]
comp["abs_delta"] = comp["delta"].abs()
comp["within_tol"] = comp["abs_delta"] <= TOL
comp["both_at_floor"] = np.isclose(comp["reported"], floor) & np.isclose(comp["recomputed"], floor)

status_tbl = comp["status"].value_counts().rename("count").to_frame()
display(status_tbl); save_table(status_tbl, "e3_computation_status")
ok = comp[comp["status"] == "ok"].copy()
print(f"\ncomputed successfully: {len(ok)} / {len(comp)}")

In [ ]:
# --- Independent cross-check: does our B-plane miss match the reported miss_distance? -------
# This validates the geometry half of the pipeline independently of the covariance half:
# if the projection were wrong, the miss distances would not agree.
sub = ok.dropna(subset=["recomputed_miss_distance", "reported_miss_distance"])
r = np.corrcoef(sub["recomputed_miss_distance"], sub["reported_miss_distance"])[0, 1]
rel = ((sub["recomputed_miss_distance"] - sub["reported_miss_distance"]).abs()
       / sub["reported_miss_distance"]).median()
print(f"projected-miss vs reported miss_distance: Pearson r = {r:.8f}, "
      f"median relative error = {rel:.3e}")
print("The encounter-plane projection reproduces the reported miss distance essentially exactly,\n"
      "so any disagreement in Pc below is attributable to the covariance/HBR half of the\n"
      "computation, not to the geometry.")

In [ ]:
# --- Figure: recomputed vs reported log-risk, with the y=x reference -----------------------
fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4))
lims = (floor - 1.5, 1.0)

for ax, drop_floor in zip(axes, (False, True)):
    d = ok[~ok["both_at_floor"]] if drop_floor else ok
    inside = d[d["within_tol"]]; outside = d[~d["within_tol"]]
    ax.plot(lims, lims, "k-", lw=1.2, label="y = x")
    ax.fill_between(lims, [lims[0] - TOL, lims[1] - TOL], [lims[0] + TOL, lims[1] + TOL],
                    color="k", alpha=0.10, label=f"±{TOL:g} log10 tolerance")
    ax.scatter(inside["reported"], inside["recomputed"], s=34, alpha=0.85,
               color="#0072B2", edgecolor="none", label=f"within tolerance (n={len(inside)})")
    ax.scatter(outside["reported"], outside["recomputed"], s=44, alpha=0.9,
               color="#D55E00", marker="^", edgecolor="none",
               label=f"outside tolerance (n={len(outside)})")
    ax.axvline(cfg.high_risk_threshold, color="grey", ls=":", lw=1.1)
    ax.set_xlim(lims); ax.set_ylim(lims)
    ax.set_xlabel("reported risk  [log10 Pc]")
    ax.set_ylabel("recomputed Pc  [log10]")
    ax.set_title("excluding floor-to-floor matches" if drop_floor
                 else "full stratified sample")
    ax.legend(frameon=False, fontsize=8, loc="upper left")

fig.suptitle("E3  Recomputed vs reported collision probability (stratified train sample)",
             y=1.00, fontsize=11)
save_fig(fig, "e3_recomputed_vs_reported_scatter")
plt.show()

## 5. Error summary, statistical tests, and bootstrap CIs

In [ ]:
# --- Error summary table --------------------------------------------------------------------
def err_block(d, name):
    if len(d) == 0:
        return {"subset": name, "n": 0}
    return {"subset": name, "n": len(d),
            "prop. within tol": float(d["within_tol"].mean()),
            "MAE [log10]": float(d["abs_delta"].mean()),
            "median |Δ|": float(d["abs_delta"].median()),
            "median Δ (bias)": float(d["delta"].median()),
            "90th pct |Δ|": float(d["abs_delta"].quantile(0.90)),
            "max |Δ|": float(d["abs_delta"].max())}

blocks = [err_block(ok, "all computed"),
          err_block(ok[~ok["both_at_floor"]], "excluding floor-to-floor matches")]
for s in ok["stratum"].unique():
    blocks.append(err_block(ok[ok["stratum"] == s], f"stratum {s}"))
err_tbl = pd.DataFrame(blocks).set_index("subset")
display(err_tbl.round(4)); save_table(err_tbl, "e3_error_summary")

In [ ]:
# --- Paired tests + bootstrap CIs (event-level resampling of the sample) --------------------
d_all = ok["delta"].to_numpy()
d_nf = ok.loc[~ok["both_at_floor"], "delta"].to_numpy()

tests = []
for name, d in (("all computed", d_all), ("excluding floor-to-floor", d_nf)):
    nz = d[~np.isclose(d, 0.0)]
    w = stats.wilcoxon(nz) if len(nz) > 0 else None
    sub = ok if name == "all computed" else ok[~ok["both_at_floor"]]
    pear = np.corrcoef(sub["reported"], sub["recomputed"])[0, 1]
    spear = stats.spearmanr(sub["reported"], sub["recomputed"]).statistic
    tests.append({"subset": name, "n": len(d), "n non-zero Δ": len(nz),
                  "Wilcoxon p": float(w.pvalue) if w else np.nan,
                  "Pearson r": float(pear), "Spearman rho": float(spear)})
tests_tbl = pd.DataFrame(tests).set_index("subset")
display(tests_tbl); save_table(tests_tbl, "e3_paired_tests")

# Bootstrap CIs on MAE and proportion-within-tolerance.
boot_rng = np.random.default_rng(cfg.seed)
B = cfg.bootstrap.n_resamples

def boot_ci(values, stat, n_boot=B):
    values = np.asarray(values, dtype=float)
    n = len(values)
    idx = boot_rng.integers(0, n, size=(n_boot, n))
    draws = np.array([stat(values[i]) for i in idx])
    return float(stat(values)), float(np.percentile(draws, 2.5)), float(np.percentile(draws, 97.5))

ci_rows = []
for name, sub in (("all computed", ok), ("excluding floor-to-floor", ok[~ok["both_at_floor"]])):
    mae, lo1, hi1 = boot_ci(sub["abs_delta"], np.mean)
    prop, lo2, hi2 = boot_ci(sub["within_tol"].astype(float), np.mean)
    ci_rows.append({"subset": name, "n": len(sub),
                    "MAE": mae, "MAE 95% CI": f"[{lo1:.3f}, {hi1:.3f}]",
                    "prop. within tol": prop, "prop. 95% CI": f"[{lo2:.3f}, {hi2:.3f}]"})
ci_tbl = pd.DataFrame(ci_rows).set_index("subset")
display(ci_tbl.round(4)); save_table(ci_tbl, "e3_bootstrap_cis")

In [ ]:
# --- Failure-mode characterisation ----------------------------------------------------------
out = ok[~ok["within_tol"]].copy()
print(f"CDMs outside the ±{TOL:g} tolerance: {len(out)} of {len(ok)}\n")
if len(out):
    out["direction"] = np.where(out["delta"] > 0, "recomputed HIGHER than reported",
                                                  "recomputed LOWER than reported")
    display(out["direction"].value_counts().rename("count").to_frame())
    display(out.groupby("stratum")["abs_delta"].agg(["count", "median", "max"]).round(3))
    cols = ["stratum", "reported", "recomputed", "delta", "time_to_tca"]
    display(out.sort_values("abs_delta", ascending=False)[cols].head(15).round(3))
    save_table(out[cols], "e3_outside_tolerance_cases")
else:
    print("no cases outside tolerance")

## 6. Assumption A4 — the measurement, stated plainly

Per `CLAUDE.md` §3 and the E3 brief, this notebook **reports** where the sample falls relative to
the pre-registered (still PROPOSED) tolerance. It does not issue the go/no-go on A4, and it does
not adjust the tolerance to the result.

In [ ]:
# --- A4 status against the pre-registered tolerance -----------------------------------------
prop_all = float(ok["within_tol"].mean())
nf = ok[~ok["both_at_floor"]]
prop_nf = float(nf["within_tol"].mean())

def band(p):
    if p >= MINPROP: return f"AT OR ABOVE the pre-registered {MINPROP:.0%} bar"
    if p >= 0.50:    return f"in the 50%-{MINPROP:.0%} 'partial hold' band"
    return "BELOW 50% — the pre-registered partial-hold band"

print("=" * 78)
print("E3 RESULT vs PRE-REGISTERED (PROPOSED) TOLERANCE")
print("=" * 78)
print(f"  per-CDM criterion      : |Δ log10 risk| <= {TOL:g}")
print(f"  sample-level bar       : >= {MINPROP:.0%} within tolerance")
print()
print(f"  full stratified sample : {prop_all:.1%} within tolerance  (n={len(ok)})")
print(f"                           -> {band(prop_all)}")
print(f"  excl. floor-to-floor   : {prop_nf:.1%} within tolerance  (n={len(nf)})")
print(f"                           -> {band(prop_nf)}")
print()
print(f"  MAE (all)              : {ok['abs_delta'].mean():.4f} log10 units")
print(f"  median |Δ| (all)       : {ok['abs_delta'].median():.4f} log10 units")
print(f"  MAE (excl. floor)      : {nf['abs_delta'].mean():.4f} log10 units")
print(f"  median |Δ| (excl.)     : {nf['abs_delta'].median():.4f} log10 units")
print("=" * 78)

a4 = pd.DataFrame([
    {"quantity": "proportion within tolerance (full sample)", "value": prop_all},
    {"quantity": "proportion within tolerance (excl. floor-to-floor)", "value": prop_nf},
    {"quantity": "pre-registered bar", "value": MINPROP},
    {"quantity": "MAE log10 (full sample)", "value": float(ok["abs_delta"].mean())},
    {"quantity": "median |Δ| log10 (full sample)", "value": float(ok["abs_delta"].median())},
    {"quantity": "computed / sampled", "value": f"{len(ok)}/{len(comp)}"},
]).set_index("quantity")
display(a4); save_table(a4, "e3_a4_status")

In [ ]:
# --- Closing statement ----------------------------------------------------------------------
print(f"""
A4 FEASIBILITY — WHAT THE SPIKE SHOWS (measurement only)

 * The Foster-type computation reproduces its analytic toy geometries to a worst-case relative
   error of {toy['rel. error'].max():.1e}, and reproduces the reported miss_distance from the state
   vectors essentially exactly (Pearson r = {r:.6f}). The engine itself is behaving.

 * The public fields ARE sufficient to run the computation: only {n_required_missing.mean():.4%} of
   training CDMs lack any field the computation requires, so field availability is not the
   binding constraint (this is the field-availability half of Q-DATA-05).

 * On the pre-declared stratified sample the central tendency is tight — median absolute
   disagreement {ok['abs_delta'].median():.3f} log10 units — but the distribution has a tail:
   {prop_all:.1%} of the full sample and {prop_nf:.1%} of the non-floor sample fall within the
   pre-registered ±{TOL:g}. The failure-mode table above characterises where the tail sits.

 * The spike-level approximations that could explain the tail are documented in
   pc_foster.py and were declared before the run: covariances summed without an inter-object
   frame rotation, HBR taken as (t_span + c_span)/2, and the position block only.

WHAT THIS NOTEBOOK DOES NOT DO (CLAUDE.md §3, §10, §11):
 * It does not declare A4 held, partially held, or failed.
 * It does not confirm, relax, or tighten the tolerance — that entry in DECISIONS.md is still
   marked PROPOSED and is Sidh's to confirm or revise.
 * It does not choose the Q-DATA-05 missing-field policy or the M7 scope that follows from it.
""")

(cfg.path("reports_dir") / "00b_pc_spike_provenance.json").write_text(
    json.dumps(PROVENANCE, indent=2), encoding="utf-8")
comp.to_csv(TABDIR / "e3_full_comparison.csv", index=False)
print("provenance sidecar : reports/00b_pc_spike_provenance.json")
print("full comparison    : reports/tables/e3_full_comparison.csv")